# The Bayesian Finite Element Method in Inverse Problems: Three-point Bending Test

This notebook is associated with section 3.2 of "The Bayesian Finite Element Method in Inverse Problems: a Critical Comparison between Probabilistic Models for Discretization Error" by Anne Poot, Iuri Rocha, Pierre Kerfriden and Frans van der Meer ([doi:10.48550/arXiv.2506.02815](https://doi.org/10.48550/arXiv.2506.02815)).

In [ ]:
# general imports
import os
import numpy as np
import urllib.request
import zipfile

# local imports
from bfem.observation import compute_bfem_observations
from fem.jive import CJiveRunner
from fem.meshing import mesh_interval_with_line2, create_phi_from_globdat, calc_elem_sizes, calc_boundary_nodes
from probability.process import GaussianProcess, InverseCovarianceOperator, ProjectedPrior
from rmfem.perturbation import calc_perturbed_coords
from util.io import read_csv_from

from experiments.reproduction.inverse.pullout_bar.props import get_fem_props
from experiments.reproduction.inverse.three_point_hole.plots import sample_plot, marginal_plot

## Inverse Problem

We now consider the inverse problem, described in section 3.2.1.
The dataset can be regenerated, but it it easier to download it directly.

In [ ]:
cwd = os.getcwd()
tmp_path = os.path.join(cwd, "tmp")
zip_path = os.path.join(tmp_path, "three-point-hole.zip")
output_path = os.path.join(tmp_path, "output")
url =  "https://data.4tu.nl/file/a610235b-7e45-4d8f-8a0b-64d8eb157b36/65baa0ae-3fd7-42b5-acbf-eacbf16e4a85"

if not os.path.exists(tmp_path):
    # download zip file
    os.mkdir(tmp_path)
    out = urllib.request.urlretrieve(url, zip_path)

    # extract in tmp folder
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(tmp_path)

assert os.path.isfile(zip_path)
assert os.path.isdir(output_path)

### Figure 3.5: posterior samples

These figures show samples from the FEM, BFEM, RM-FEM and statFEM posteriors of the inverse problem from section 3.2.1 of the paper.

In [ ]:
# rng seed
# seed = 0        # single run
seed = "0-20"     # meta run

if isinstance(seed, int):
    # for the standard run, a burn-in period is needed during which the proposal is adapted
    N_burn = 10000
    N_filter = 200
elif isinstance(seed, str):
    # for the meta run, the proposal is never adapted, so no burn-in period is needed
    N_burn = 100
    N_filter = 200
else:
    assert False

for fem_type in ["fem", "bfem", "rmfem", "statfem"]:
    # load data, discard burn-in and thin samples
    fname = os.path.join(output_path, "samples-{}_seed-{}.csv".format(fem_type, seed))
    df = read_csv_from(fname, "x,y,a,theta,r_rel")
    df = df[(df["sample"] >= N_burn) & (df["sample"] % N_filter == 0)]
    df = df[df["h"].isin([0.2, 0.1, 0.05])]
    df = df[df["std_corruption"] == 1e-4]

    # plot the samples
    sample_plot(df)

### Figure 3.6: posterior marginals

This figure shows the marginal distribution of each parameter for the FEM, BFEM, RM-FEM and statFEM posteriors of the inverse problem from section 3.2.1 of the paper.

In [ ]:
# rng seed
# seed = 0        # single run
seed = "0-20"     # meta run

if isinstance(seed, int):
    # for the standard run, a burn-in period is needed during which the proposal is adapted
    N_burn = 10000
    N_filter = 200
elif isinstance(seed, str):
    # for the meta run, the proposal is never adapted, so no burn-in period is needed
    N_burn = 100
    N_filter = 200
else:
    assert False

dfs = {}

for fem_type in ["fem", "bfem", "rmfem", "statfem"]:
    # load data, discard burn-in and thin samples
    fname = os.path.join(output_path, "samples-{}_seed-{}.csv".format(fem_type, seed))
    df = read_csv_from(fname, "x,y,a,theta,r_rel")
    df = df[(df["sample"] >= N_burn) & (df["sample"] % N_filter == 0)]
    df = df[df["h"].isin([0.2, 0.1, 0.05])]
    df = df[df["std_corruption"] == 1e-4]
    df = df[["x", "a", "theta", "r_rel", "h"]]

    # add dataframe to dictionary
    dfs[fem_type] = df
    
marginal_plot(dfs)

### Figure 3.8: statFEM hyperparameter posterior marginals
This figure shows the marginal distribution of each hyperparameter for the statFEM posterior of the inverse problem from section 3.2.1 of the paper.

In [ ]:
# rng seed
# seed = 0        # single run
seed = "0-20"     # meta run

if isinstance(seed, int):
    # for the standard run, a burn-in period is needed during which the proposal is adapted
    N_burn = 10000
    N_filter = 200
elif isinstance(seed, str):
    # for the meta run, the proposal is never adapted, so no burn-in period is needed
    N_burn = 100
    N_filter = 200
else:
    assert False

dfs = {}

for fem_type in ["statfem"]:
    # load data, discard burn-in and thin samples
    fname = os.path.join(output_path, "samples-{}_seed-{}.csv".format(fem_type, seed))
    df = read_csv_from(fname, "x,y,a,theta,r_rel")
    df = df[(df["sample"] >= N_burn) & (df["sample"] % N_filter == 0)]
    df = df[df["h"].isin([0.2, 0.1, 0.05])]
    df = df[df["std_corruption"] == 1e-4]
    df = df[["rho", "log_l_d", "log_sigma_d", "h"]]

    # add dataframe to dictionary
    dfs[fem_type] = df
    
marginal_plot(dfs)